# Ambito delle voci

Questo notebook analizza l'**ambito delle voci** (estensione, *ambitus*) di una o più composizioni. L'analisi si concentra su alcuni madrigali di Luca Marenzio.

---

### Madrigali di Luca Marenzio

Il notebook approfondisce i madrigali di Luca Marenzio inclusi nel *Quarto libro de' madrigali a sei voci* (Venezia, Giacomo Vincenti, 1587, RISM A/1 M 510) e nel *Sesto libro de madrigali a cinque voci* (Venezia, Angelo Gardano, 1594, RISM A/1 M 557). Per l'analisi vengono impiegate le partiture in formato `.mei` della **[Marenzio Online Digital Edition (MODE)](http://www.marenzio.org):**

- Luca Marenzio, *Il quarto libro de' madrigali a sei voci*, a cura di Lucia Marchi,
  [MODE – Marenzio Online Digital Edition](https://marenzio.org/digital-edition/M-04-6.xhtml), 2024 (15 files);

- Luca Marenzio, *Il sesto libro de madrigali a cinque voci*, a cura di Daniele Filippi,
  [MODE – Marenzio Online Digital Edition](https://marenzio.org/digital-edition/M-06-5.xhtml), 2019 (17 files).

I file `.mei` sono stati scaricati da [github.com/marenzio/marenzio.github.io](https://github.com/marenzio/marenzio.github.io) e sottoposti a una minima preparazione tecnica per la piena compatibilità con `crim-intervals`.
Per i dettagli vedere la sezione
[*Marenzio Corpus: Sources and Credits*](https://github.com/gtaschetti/analisi_mus_unipd#marenzio-corpus-sources-and-credits) del README di questo repository.

### Librerie

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from crim_intervals import importScore

print('Ok')

### Costanti e funzioni di supporto

La cella seguente carica alcune funzioni e costanti di supporto per la corretta esecuzione del codice di questo notebook. In particolare, è necessario indicare l'ordine della gamma dei suoni (attraverso `pitch_order`), ossia dal grave all'acuto, per evitare che le note vengano ordinate alfabeticamente (procedura standard).
- `pitch_order` / `pitch_to_num` — mappa altezze → numero ordinale (Eb2..B5).
- `get_voice_range(piece)` — `{voce: (nota_min, nota_max)}`.
- `plot_voice_ranges(ranges, title)` — grafico a barre orizzontali (matplotlib).

In [ ]:
import re

pitch_order = [
    'C2', 'C#2', 'D2', 'E-2', 'E2', 'F2', 'F#2', 'G2', 'A2', 'B-2', 'B2',
    'C3', 'C#3', 'D3', 'E-3', 'E3', 'F3', 'F#3', 'G3', 'G#3', 'A3', 'B-3', 'B3',
    'C4', 'C#4', 'D4', 'E-4', 'E4', 'F4', 'F#4', 'G4', 'A4', 'B-4', 'B4',
    'C5', 'C#5', 'D5', 'E-5', 'E5', 'F5', 'F#5', 'G5', 'A5', 'B-5', 'B5',
    'C6', 'C#6', 'D6', 'E-6', 'E6', 'F6', 'F#6', 'G6', 'A6', 'B-6', 'B6',
]
pitch_to_num = {p: i for i, p in enumerate(pitch_order)}

_ORD = r'(?:VI{0,2}|IV|V?I{1,3}|V|X|prim[aeo]?|second[aeo]?|terz[ao]|quart[ao]|quint[ao]|sest[ao]|primus|secundus|tertius|quartus|quintus|sextus|\d+)'
_BASE_VOICES = ['Canto', 'Alto', 'Contralto', 'Tenore', 'Basso', 'Soprano', 'Baritono']

def normalize_voice(name):
    """'Canto Secondo', 'Tenore II', 'Primo Basso', 'Canto 1' → base voice name; others unchanged."""
    for base in _BASE_VOICES:
        if re.fullmatch(rf'{re.escape(base)}\s+{_ORD}.*', name, re.IGNORECASE) or \
           re.fullmatch(rf'{_ORD}\s+{re.escape(base)}', name, re.IGNORECASE):
            return base
    return name


def get_voice_range(piece):
    notes = piece.notes().fillna('-')
    result = {}
    for voice in notes.columns:
        ps = [p for p in notes[voice] if p not in ('-', 'Rest') and p in pitch_to_num]
        if ps:
            result[voice] = (
                min(ps, key=lambda p: pitch_to_num[p]),
                max(ps, key=lambda p: pitch_to_num[p]),
            )
    return result


def plot_voice_ranges(ranges, title):
    voices = list(ranges.keys())
    lo_nums = [pitch_to_num[ranges[v][0]] for v in voices]
    hi_nums = [pitch_to_num[ranges[v][1]] for v in voices]
    all_lo, all_hi = min(lo_nums), max(hi_nums)

    fig, ax = plt.subplots(figsize=(12, len(voices) * 0.7 + 1.5))
    for i, voice in enumerate(voices):
        lo_n = pitch_to_num[ranges[voice][0]]
        hi_n = pitch_to_num[ranges[voice][1]]
        ax.barh(i, hi_n - lo_n, left=lo_n, alpha=0.75, height=0.6)
        ax.text(lo_n - 0.3, i, ranges[voice][0], ha='right', va='center', fontsize=8)
        ax.text(hi_n + 0.3, i, ranges[voice][1], ha='left',  va='center', fontsize=8)

    tick_pos = list(range(all_lo, all_hi + 1))
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([pitch_order[p] for p in tick_pos], rotation=70, fontsize=7)
    ax.set_yticks(range(len(voices)))
    ax.set_yticklabels(voices, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=11, pad=10)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(all_lo - 2, all_hi + 2)
    plt.tight_layout()
    plt.show()


print('Funzioni caricate.')

---
## Sezione 1 -- Analisi di una singola composizione

Carica una composizione indicando il `filename`.
Se il caricamento è andato a buon fine, `print(piece.metadata)` restituisce titolo e compositore.

In [ ]:
filename = 'Marenzio_o_voi.xml'   # <- cambia qui
piece = importScore(f'Music_Files/{filename}')
print(piece.metadata)

### Tabella delle note

`piece.notes()`: ogni **riga** e' un offset temporale (semiminima), ogni **colonna** e' una voce, ogni **cella** e' un'altezza o una pausa (`'Rest'`).
Le celle vuote (`NaN`) indicano che in quel punto viene prolungato il suono (o la pausa) precedente.
Il comando `.head(20)` consente di mostrare soltanto le prime righe della tabella (*head* = testa).
Il numero delle righe visualizzate può essere modificato cambiando il numero fra parentesi.

In [ ]:
piece.notes().fillna('').head(20)

### Conteggio e distribuzione delle altezze

Quante volte compare ogni altezza in ciascuna voce? La tabella seguente mostra, per tutte le voci (colonne), il numero delle occorrenze di ciascuna nota. Le note sono ordinate dalla più grave alla più acuta grazie alla costante `pitch_order` che è stata definita precedentemente. Il grafico a barre sovrapposte li rappresenta questo dato visivamente.

*Questa visualizzazione è presa dai notebooks del [CRIM Project](https://crimproject.org) — Haverford College, PA.*

In [ ]:
# Conteggio delle altezze per voce, ordinato dal grave all'acuto
# Adattato da: CRIM Project (https://crimproject.org), Haverford College, PA
_nr = piece.notes()
pitch_counts = (
    _nr.apply(pd.Series.value_counts)
       .fillna(0)
       .astype(int)
       .reset_index()
       .rename(columns={'index': 'pitch'})
)
pitch_counts['pitch'] = pd.Categorical(pitch_counts['pitch'], categories=pitch_order)
pitch_counts = pitch_counts.sort_values('pitch').dropna().set_index('pitch')
pitch_counts

In [ ]:
md = piece.metadata
ax = pitch_counts.plot(kind='bar', stacked=True, figsize=(14, 5))
ax.set_title(f"Distribuzione delle altezze — {md['composer']}, {md['title']}", pad=10)
ax.set_xlabel('Altezza')
ax.set_ylabel('Conteggio')
ax.legend(title='Voce', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Estensione vocale

La seguente tabella riporta l'estensione (*ambitus*) di ciascuna voce. Per ogni **voce** vengono indicate la nota più grave (**min**) e la nota più acuta (**max**).

In [ ]:
ranges = get_voice_range(piece)
rows = [
    {'voce': v, 'min': lo, 'max': hi}
    for v, (lo, hi) in ranges.items()
]
pd.DataFrame(rows)

I dati raccolti con il codice appena eseguito possono essere rappresentati graficamente. Ogni ambito vocale viene qui rappresentato come una barra che si estende orizzontalmente da sinistra a destra lungo la gamma dei suoni (dal più grave al più acuto).

In [ ]:
md = piece.metadata
plot_voice_ranges(ranges, title=f"Estensioni - {md['composer']}, {md['title']}")

---
## Sezione 2 — Selezione del corpus

Nella seconda parte di questo notebook è possibile analizzare un *corpus* di composizioni.
Le celle seguenti definiscono alcune **liste predefinite** che costituiscono il *corpus* dei files da analizzare.
Esegui ciascuna cella per vedere il contenuto della lista corrispondente, poi scegli quale usare nell'ultima cella di questa sezione, modificando la variabile `lista_selezionata`.

In [ ]:
# Marenzio, Quarto libro a 6 voci
lista_libro_IV_6v = sorted(str(p) for p in Path('Music_Files').glob('M_04_6_*.mei'))

print(f'lista_libro_IV_6v — {len(lista_libro_IV_6v)} composizioni:')
lista_libro_IV_6v

In [ ]:
# Marenzio, Sesto libro a 5 voci
lista_libro_VI_5v = sorted(str(p) for p in Path('Music_Files').glob('M_06_5_*.mei'))

print(f'lista_libro_VI_5v — {len(lista_libro_VI_5v)} composizioni:')
lista_libro_VI_5v

In [ ]:
# Entrambi i libri di Marenzio (Quarto libro a 6, Sesto libro a 5)
lista_marenzio = sorted(str(p) for p in Path('Music_Files').glob('M_*.mei'))

print(f'lista_marenzio — {len(lista_marenzio)} composizioni:')
lista_marenzio

In [ ]:
# tutti i files nella cartella tranne i due libri di Marenzio (quarto a 6, Sesto a 5)
no_marenzio = sorted(
    str(p) for p in Path('Music_Files').iterdir()
    if p.suffix in ('.xml', '.musicxml', '.mxl', '.mei')
    and not p.name.startswith(('M_04_6_', 'M_06_5_'))
)

print(f'no_marenzio — {len(no_marenzio)} composizioni:')
no_marenzio

Per analizzare un insieme personalizzato di file, inserisci i percorsi nella lista seguente ed eseguila. Inserisci anche il prefisso 'Music_Files/' prima del nome di ciascun file.

In [ ]:
lista_personalizzata = [
    'Music_Files/Marenzio_o_voi.xml',
    'Music_Files/Monteverdi_cantai.musicxml',
    'Music_Files/Monteverdi_o_rossignuol.xml',
]

print(f'lista_personalizzata — {len(lista_personalizzata)} composizioni:')
lista_personalizzata

**Seleziona la lista** da analizzare indicando il suo nome dopo `lista_selezionata =` nella cella seguente, poi eseguila.
La lista selezionata viene stampata per verifica.

In [ ]:
lista_selezionata = lista_libro_VI_5v   # <- cambia qui il nome della lista

print(f'{len(lista_selezionata)} composizioni selezionate:')
for f in lista_selezionata:
    print(' ', Path(f).name)

---
## Sezione 3 — Corpus: estensioni vocali

Il codice carica ogni composizione della `lista_selezionata`, calcola le estensioni di ogni voce
e raccoglie i risultati all'interno del **dizionario** (termine tecnico) `all_ranges`..

In [ ]:
all_ranges = {}

for path in lista_selezionata:
    p = importScore(path)
    if p is None:
        print(f'Errore: {path}')
        continue
    meta  = p.metadata
    label = f"{meta['composer']}  -  {meta['title']}"
    all_ranges[label] = get_voice_range(p)

print(f'Caricate {len(all_ranges)} composizioni.')

Il codice seguente produce per ciascuna composizione del corpus un grafico come quello già impiegato nella Sezione 1 di questo notebook.

In [ ]:
for label, ranges in all_ranges.items():
    plot_voice_ranges(ranges, title=label)

### Riepilogo delle estensioni

La tabella seguente restituisce in forma sintetica l'ambito delle voci di ciascuna composizione del *corpus*.

In [ ]:
rows = []
for label, r in all_ranges.items():
    if not r:
        continue
    composer, title = label.split('  -  ', 1)
    row = {'composer': composer, 'title': title}
    for voice, (lo, hi) in r.items():
        row[normalize_voice(voice)] = f'{lo}–{hi}'
    rows.append(row)

pd.DataFrame(rows)

---
## Sezione 4 — Le voci "jolly": Quinto e Sesto

Nel madrigale a 5 o 6 voci il **Quinto** e il **Sesto** non corrispondono a un registro fisso: possono fungere da secondo Canto, secondo Alto, secondo Tenore o secondo Basso a seconda della composizione.

Il primo grafico evidenzia l'estensione di queste due parti vocali all'interno di ogni composizione del *corpus*.

Il secondo grafico mostrata l'estensione complessiva di ciascun tipo di voce considerando tutte le composizioni del *corpus*.

In [ ]:
from collections import defaultdict
import plotly.graph_objects as go

JOLLY = {'quinto', 'sesto'}

voice_color = {
    'Canto':  '#1f77b4',
    'Alto':   '#2ca02c',
    'Tenore': '#ff7f0e',
    'Basso':  '#d62728',
    'Quinto': '#9467bd',
    'Sesto':  '#e377c2',
}

# ── Aggregazione per voce normalizzata ───────────────────────────────────────
voice_data = defaultdict(list)
for label, ranges in all_ranges.items():
    for voice, (lo, hi) in ranges.items():
        lo_n = pitch_to_num.get(lo)
        hi_n = pitch_to_num.get(hi)
        if lo_n is None or hi_n is None:
            continue
        voice_data[normalize_voice(voice)].append((lo_n, hi_n, label))

ordered_voices = [v for v in ['Canto', 'Sesto', 'Alto', 'Quinto', 'Tenore', 'Basso']
                  if v in voice_data]

all_lo_idx = min(lo for entries in voice_data.values() for lo, hi, _ in entries)
all_hi_idx = max(hi for entries in voice_data.values() for lo, hi, _ in entries)
tick_vals = list(range(all_lo_idx, all_hi_idx + 1))
tick_text  = [pitch_order[p] for p in tick_vals]

print(f'Voci nel corpus: {ordered_voices}')
print(f'Ambitus complessivo: {pitch_order[all_lo_idx]} – {pitch_order[all_hi_idx]}')

### Quinto e Sesto in evidenza

Per ogni composizione viene generato un piccolo grafico a barre orizzontali con le voci di Quinto e Sesto in evidenza.

In [ ]:
pieces_jolly = {lbl: r for lbl, r in all_ranges.items()
                if any(normalize_voice(v).lower() in JOLLY for v in r)}

if not pieces_jolly:
    print('Nessuna composizione con voci Quinto o Sesto nel corpus caricato.')
else:
    FIG_W, FIG_H = 10, 1.6   # larghezza e altezza fisse per ogni figura (pollici)

    _cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

    def _voice_mid(ranges, v):
        lo, hi = ranges[v]
        return (pitch_to_num.get(lo, 0) + pitch_to_num.get(hi, 0)) / 2

    for lbl, ranges in pieces_jolly.items():
        voices_sorted = sorted(ranges.keys(), key=lambda v: -_voice_mid(ranges, v))
        n = len(voices_sorted)

        # assegna i colori del ciclo di default alle voci jolly, in ordine di altezza
        jolly_voices = [v for v in voices_sorted if normalize_voice(v).lower() in JOLLY]
        jolly_color  = {v: _cycle[i % len(_cycle)] for i, v in enumerate(jolly_voices)}

        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

        for i, voice in enumerate(voices_sorted):
            lo, hi = ranges[voice]
            lo_n, hi_n = pitch_to_num.get(lo), pitch_to_num.get(hi)
            if lo_n is None or hi_n is None:
                continue
            is_jolly = normalize_voice(voice).lower() in JOLLY
            color = jolly_color.get(voice, '#cccccc')
            ax.barh(i, hi_n - lo_n, left=lo_n, color=color,
                    height=0.55, alpha=0.85, linewidth=0)
            ax.text(lo_n - 0.3, i, lo, ha='right', va='center', fontsize=7,
                    color=color if is_jolly else '#aaaaaa')
            ax.text(hi_n + 0.3, i, hi, ha='left',  va='center', fontsize=7,
                    color=color if is_jolly else '#aaaaaa')

        all_lo = min(pitch_to_num[r[0]] for r in ranges.values() if r[0] in pitch_to_num)
        all_hi = max(pitch_to_num[r[1]] for r in ranges.values() if r[1] in pitch_to_num)
        tick_pos = list(range(all_lo, all_hi + 1))
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([pitch_order[p] for p in tick_pos], rotation=70, fontsize=7)
        ax.set_yticks(range(n))
        ax.set_yticklabels(voices_sorted, fontsize=8)
        ax.invert_yaxis()
        title = lbl.split('  -  ', 1)[-1] if '  -  ' in lbl else lbl
        ax.set_title(title, fontsize=9, pad=6)
        ax.set_xlim(all_lo - 2, all_hi + 2)
        ax.grid(axis='x', alpha=0.2)
        plt.tight_layout()
        plt.show()

### Estensione complessiva di ogni tipo di voce all'interno del corpus

Il grafico mostra l'estensione complessiva di ciascun tipo di voce (C, A, T, B, Q, S) considerando **tutte le composizioni del corpus selezionato**. Come si presentano le estensioni complessive del Quinto e del Sesto rispetto a quelle delle altre voci?

In [ ]:
fig_orig2 = go.Figure()

for voice in ordered_voices:
    entries = voice_data[voice]
    if not entries:
        continue
    lo = min(e[0] for e in entries)
    hi = max(e[1] for e in entries)
    color = voice_color.get(voice, '#8c564b')
    fig_orig2.add_trace(go.Scatter(
        x=[lo, hi], y=[voice, voice],
        mode='lines+markers',
        line=dict(color=color, width=10),
        marker=dict(symbol='line-ns-open', size=10, line=dict(color=color, width=2)),
        name=voice,
        showlegend=False,
        hovertemplate=f'<b>{voice}</b>: {pitch_order[lo]}–{pitch_order[hi]} ({hi - lo} st)<extra></extra>',
    ))

fig_orig2.update_layout(
    title='Estensione complessiva per voce (corpus)',
    xaxis=dict(tickvals=tick_vals, ticktext=tick_text, title='Altezza',
               range=[-1, len(pitch_order)]),
    yaxis=dict(title='', categoryorder='array',
               categoryarray=list(reversed(ordered_voices))),
    template='plotly_white',
    height=60 * len(ordered_voices) + 80,
    margin=dict(l=20, r=40, t=70, b=50),
)
fig_orig2.show()